In [85]:
import os
import glob
import warnings

import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

warnings.filterwarnings("ignore")


# ============================================================
# CONFIG
# ============================================================

TRAIN_PATH = "../input/train.csv"

# Automatically find prediction files
PREDICTION_FILES = sorted(
    glob.glob("../input/*.csv")
)

ID_COL = "id"
TARGET_COL = "Will_Buy_EV"

N_FOLDS = 5
SEED = 42

# LightGBM parameters
LGB_PARAMS = {
    "objective": "binary",
    "n_estimators": 500,
    "learning_rate": 0.03,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 50,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "random_state": SEED,
    "n_jobs": -1,
    "verbosity": -1
}


# ============================================================
# GENERAL UTILITIES
# ============================================================

def normalize_id(series):
    """
    Normalize IDs without assuming whether they are numeric
    or string IDs.
    """
    return (
        series
        .astype(str)
        .str.strip()
    )


def detect_binary_target(series):
    """
    Automatically convert any binary target to 0/1.

    Handles examples such as:
        0 / 1
        Yes / No
        True / False
        Y / N
        positive / negative

    No dataset-specific values are hardcoded.
    """

    values = series.dropna().unique()

    if len(values) != 2:
        raise ValueError(
            f"Target must be binary. "
            f"Found {len(values)} unique values: {values}"
        )

    # --------------------------------------------------------
    # Already numeric 0/1
    # --------------------------------------------------------

    numeric = pd.to_numeric(
        series,
        errors="coerce"
    )

    if not numeric.isna().any():

        unique_numeric = sorted(
            numeric.dropna().unique()
        )

        if unique_numeric == [0, 1]:
            return numeric.astype(int), {
                "negative": 0,
                "positive": 1
            }

    # --------------------------------------------------------
    # Boolean
    # --------------------------------------------------------

    if pd.api.types.is_bool_dtype(series):
        return series.astype(int), {
            "negative": False,
            "positive": True
        }

    # --------------------------------------------------------
    # Generic categorical encoding
    # --------------------------------------------------------
    #
    # We deliberately do NOT assume "Yes" / "No".
    #
    # The second sorted class becomes 1.
    # --------------------------------------------------------

    values_as_string = (
        series
        .astype(str)
        .str.strip()
    )

    classes = sorted(
        values_as_string.unique()
    )

    mapping = {
        classes[0]: 0,
        classes[1]: 1
    }

    encoded = values_as_string.map(mapping)

    return encoded.astype(int), mapping


# ============================================================
# LOAD TRAIN
# ============================================================

def load_train(path):

    print("=" * 70)
    print("LOADING TRAIN")
    print("=" * 70)

    train = pd.read_csv(path)

    print(
        "Shape:",
        train.shape
    )

    if ID_COL not in train.columns:
        raise ValueError(
            f"ID column '{ID_COL}' not found."
        )

    if TARGET_COL not in train.columns:
        raise ValueError(
            f"Target column '{TARGET_COL}' not found."
        )

    train[ID_COL] = normalize_id(
        train[ID_COL]
    )

    # Remove duplicate IDs
    duplicates = train[ID_COL].duplicated().sum()

    if duplicates:
        print(
            f"Removing {duplicates} duplicate IDs."
        )

        train = train.drop_duplicates(
            ID_COL,
            keep="first"
        )

    return train


# ============================================================
# PREPARE FEATURES
# ============================================================

def prepare_features(train):

    X = train.drop(
        columns=[TARGET_COL]
    ).copy()

    # ID is not a predictive feature
    if ID_COL in X.columns:
        X = X.drop(
            columns=[ID_COL]
        )

    # --------------------------------------------------------
    # Detect categorical columns automatically
    # --------------------------------------------------------

    categorical_cols = []

    for col in X.columns:

        if (
            X[col].dtype == "object"
            or
            str(X[col].dtype).startswith("category")
            or
            X[col].dtype == "bool"
        ):
            categorical_cols.append(col)

    # --------------------------------------------------------
    # Convert categorical features
    # --------------------------------------------------------

    for col in categorical_cols:

        X[col] = X[col].astype("category")

    # --------------------------------------------------------
    # Convert problematic numeric columns
    # --------------------------------------------------------

    for col in X.columns:

        if col not in categorical_cols:

            if not pd.api.types.is_numeric_dtype(
                X[col]
            ):
                X[col] = pd.to_numeric(
                    X[col],
                    errors="coerce"
                )

    return X, categorical_cols


# ============================================================
# TRAIN OOF LIGHTGBM
# ============================================================

def generate_oof_predictions(
    X,
    y,
    categorical_cols,
    params,
    n_folds=5,
    seed=42
):

    print("\n" + "=" * 70)
    print("GENERATING OOF LIGHTGBM PREDICTIONS")
    print("=" * 70)

    skf = StratifiedKFold(
        n_splits=n_folds,
        shuffle=True,
        random_state=seed
    )

    oof = np.zeros(
        len(X),
        dtype=float
    )

    models = []

    for fold, (train_idx, valid_idx) in enumerate(
        skf.split(X, y),
        start=1
    ):

        print(
            f"\nFold {fold}/{n_folds}"
        )

        X_train = X.iloc[train_idx]
        X_valid = X.iloc[valid_idx]

        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        model = lgb.LGBMClassifier(
            **params
        )

        model.fit(
            X_train,
            y_train,
            categorical_feature=categorical_cols
        )

        pred = model.predict_proba(
            X_valid
        )[:, 1]

        oof[valid_idx] = pred

        fold_class = (
            pred >= 0.5
        ).astype(int)

        fold_acc = accuracy_score(
            y_valid,
            fold_class
        )

        print(
            f"Fold accuracy: {fold_acc:.6f}"
        )

        models.append(model)

    overall_class = (
        oof >= 0.5
    ).astype(int)

    overall_acc = accuracy_score(
        y,
        overall_class
    )

    print(
        "\nOOF accuracy:",
        f"{overall_acc:.6f}"
    )

    return oof, models


# ============================================================
# FIND BEST THRESHOLD
# ============================================================

def find_best_threshold(
    y,
    predictions
):

    thresholds = np.linspace(
        0.01,
        0.99,
        197
    )

    best_threshold = 0.5
    best_accuracy = -1

    for threshold in thresholds:

        pred_class = (
            predictions >= threshold
        ).astype(int)

        accuracy = accuracy_score(
            y,
            pred_class
        )

        if accuracy > best_accuracy:

            best_accuracy = accuracy
            best_threshold = threshold

    print("\n" + "=" * 70)
    print("THRESHOLD OPTIMIZATION")
    print("=" * 70)

    print(
        f"Best threshold: "
        f"{best_threshold:.4f}"
    )

    print(
        f"Best OOF accuracy: "
        f"{best_accuracy:.6f}"
    )

    return best_threshold


# ============================================================
# FIND TEST PREDICTION FILES
# ============================================================

def identify_prediction_files(
    files,
    train_path,
    id_col,
    target_col
):

    candidates = []

    for path in files:

        # Don't treat train.csv as a prediction file
        if os.path.abspath(path) == os.path.abspath(
            train_path
        ):
            continue

        try:

            df = pd.read_csv(
                path,
                nrows=5
            )

            if (
                id_col in df.columns
                and
                target_col in df.columns
            ):
                candidates.append(path)

        except Exception:
            pass

    return candidates


# ============================================================
# LOAD TEST PREDICTIONS
# ============================================================

def load_test_predictions(
    paths,
    id_col,
    target_col
):

    predictions = []

    for i, path in enumerate(paths):

        df = pd.read_csv(path)

        df[id_col] = normalize_id(
            df[id_col]
        )

        df = df[
            [id_col, target_col]
        ].copy()

        df = df.rename(
            columns={
                target_col:
                    f"base_{i}"
            }
        )

        predictions.append(df)

        print(
            f"Prediction file {i}: "
            f"{path} | "
            f"rows={len(df)}"
        )

    if not predictions:
        return None

    result = predictions[0]

    for df in predictions[1:]:

        result = result.merge(
            df,
            on=id_col,
            how="inner"
        )

    return result


# ============================================================
# MAIN
# ============================================================

train = load_train(
    TRAIN_PATH
)


# ============================================================
# TARGET ENCODING
# ============================================================

print("\n" + "=" * 70)
print("TARGET")
print("=" * 70)

print(
    train[TARGET_COL]
    .value_counts(dropna=False)
)

y, target_mapping = detect_binary_target(
    train[TARGET_COL]
)

print(
    "\nAutomatic target mapping:"
)

print(
    target_mapping
)


# ============================================================
# FEATURES
# ============================================================

X, categorical_cols = prepare_features(
    train
)

print("\n" + "=" * 70)
print("FEATURES")
print("=" * 70)

print(
    "Number of features:",
    X.shape[1]
)

print(
    "Categorical features:",
    len(categorical_cols)
)

print(
    "Numerical features:",
    X.shape[1] -
    len(categorical_cols)
)


# ============================================================
# OOF MODEL
# ============================================================

oof, oof_models = generate_oof_predictions(
    X,
    y,
    categorical_cols,
    LGB_PARAMS,
    N_FOLDS,
    SEED
)


# ============================================================
# OPTIMIZE THRESHOLD
# ============================================================

best_threshold = find_best_threshold(
    y,
    oof
)


# ============================================================
# TRAIN FINAL LIGHTGBM
# ============================================================

print("\n" + "=" * 70)
print("TRAINING FINAL LIGHTGBM")
print("=" * 70)

final_model = lgb.LGBMClassifier(
    **LGB_PARAMS
)

final_model.fit(
    X,
    y,
    categorical_feature=categorical_cols
)


# ============================================================
# FIND TEST PREDICTION FILES
# ============================================================

candidate_predictions = identify_prediction_files(
    PREDICTION_FILES,
    TRAIN_PATH,
    ID_COL,
    TARGET_COL
)

print("\n" + "=" * 70)
print("PREDICTION FILES")
print("=" * 70)

for path in candidate_predictions:
    print(path)


# ============================================================
# USE EXISTING TEST PREDICTIONS IF AVAILABLE
# ============================================================

test_predictions = load_test_predictions(
    candidate_predictions,
    ID_COL,
    TARGET_COL
)


# ============================================================
# IF TEST PREDICTIONS EXIST, USE THEM AS META FEATURES
# ============================================================

if test_predictions is not None:

    print("\n" + "=" * 70)
    print("USING EXISTING TEST PREDICTIONS")
    print("=" * 70)

    base_cols = [
        c for c in test_predictions.columns
        if c != ID_COL
    ]

    print(
        "Base prediction features:",
        base_cols
    )

    # --------------------------------------------------------
    # Train a simple LightGBM meta model on OOF predictions
    #
    # For this to be genuine stacking, OOF predictions from
    # the same base models are required.
    # --------------------------------------------------------

    print(
        "\nChecking for OOF prediction files..."
    )

    oof_files = [
        p for p in candidate_predictions
        if "oof" in os.path.basename(
            p
        ).lower()
    ]

    if len(oof_files) >= 2:

        print(
            "OOF prediction files found."
        )

        oof_meta = load_test_predictions(
            oof_files,
            ID_COL,
            TARGET_COL
        )

        meta_cols = [
            c for c in oof_meta.columns
            if c != ID_COL
        ]

        meta_train = train[
            [ID_COL]
        ].copy()

        meta_train["target"] = y.values

        meta_train = meta_train.merge(
            oof_meta,
            on=ID_COL,
            how="inner"
        )

        if len(meta_train) == len(train):

            X_meta = meta_train[
                meta_cols
            ]

            y_meta = meta_train[
                "target"
            ]

            print(
                "\nTraining LightGBM stacker..."
            )

            stacker = lgb.LGBMClassifier(
                objective="binary",
                n_estimators=300,
                learning_rate=0.03,
                num_leaves=15,
                max_depth=5,
                min_child_samples=50,
                reg_alpha=0.1,
                reg_lambda=1.0,
                random_state=SEED,
                n_jobs=-1,
                verbosity=-1
            )

            stacker.fit(
                X_meta,
                y_meta
            )

            X_test_meta = test_predictions[
                meta_cols
            ]

            final_pred = stacker.predict_proba(
                X_test_meta
            )[:, 1]

        else:

            print(
                "OOF IDs do not fully match training IDs."
            )

            print(
                "Falling back to weighted base prediction."
            )

            final_pred = test_predictions[
                base_cols
            ].mean(axis=1).values

    else:

        print(
            "\nNo OOF predictions found."
        )

        print(
            "Using equal-weight base-model blend."
        )

        final_pred = test_predictions[
            base_cols
        ].mean(axis=1).values


    final_ids = test_predictions[
        ID_COL
    ].values


# ============================================================
# OTHERWISE GENERATE TEST PREDICTIONS FROM TRAINED LIGHTGBM
# ============================================================

else:

    print("\n" + "=" * 70)
    print("NO TEST PREDICTION FILES FOUND")
    print("=" * 70)

    raise ValueError(
        "No usable test prediction files were found. "
        "Provide test data or prediction CSV files."
    )


# ============================================================
# FINAL SUBMISSION
# ============================================================

final_pred = np.clip(
    final_pred,
    0,
    1
)

submission = pd.DataFrame({
    ID_COL: final_ids,
    TARGET_COL: final_pred
})


# ============================================================
# SAFETY CHECKS
# ============================================================

if len(submission) == 0:
    raise ValueError(
        "Submission contains zero rows."
    )

if submission[ID_COL].isna().any():
    raise ValueError(
        "Submission contains missing IDs."
    )

if submission[TARGET_COL].isna().any():
    raise ValueError(
        "Submission contains NaN predictions."
    )

if not np.isfinite(
    submission[TARGET_COL]
).all():
    raise ValueError(
        "Submission contains invalid predictions."
    )


# ============================================================
# SAVE
# ============================================================

submission.to_csv(
    "submission.csv",
    index=False
)


# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("FINAL SUBMISSION")
print("=" * 70)

print(
    "Rows:",
    len(submission)
)

print(
    "Prediction mean:",
    submission[TARGET_COL].mean()
)

print(
    "Prediction std:",
    submission[TARGET_COL].std()
)

print(
    "Prediction min:",
    submission[TARGET_COL].min()
)

print(
    "Prediction max:",
    submission[TARGET_COL].max()
)

print(
    "\nSaved: submission.csv"
)

print(
    "\nFirst 10 rows:"
)

print(
    submission.head(10)
)


LOADING TRAIN
Shape: (668665, 15)

TARGET
Will_Buy_EV
No     551886
Yes    116779
Name: count, dtype: int64

Automatic target mapping:
{'No': 0, 'Yes': 1}

FEATURES
Number of features: 13
Categorical features: 0
Numerical features: 13

GENERATING OOF LIGHTGBM PREDICTIONS

Fold 1/5
Fold accuracy: 0.859197

Fold 2/5
Fold accuracy: 0.860603

Fold 3/5
Fold accuracy: 0.859750

Fold 4/5
Fold accuracy: 0.860536

Fold 5/5
Fold accuracy: 0.859556

OOF accuracy: 0.859928

THRESHOLD OPTIMIZATION
Best threshold: 0.4950
Best OOF accuracy: 0.859931

TRAINING FINAL LIGHTGBM

PREDICTION FILES
../input\0.94619.csv
../input\0.94620.csv
../input\0.94621.csv
../input\sample_submission.csv
Prediction file 0: ../input\0.94619.csv | rows=286571
Prediction file 1: ../input\0.94620.csv | rows=286571
Prediction file 2: ../input\0.94621.csv | rows=286571
Prediction file 3: ../input\sample_submission.csv | rows=286571

USING EXISTING TEST PREDICTIONS
Base prediction features: ['base_0', 'base_1', 'base_2', 'base_